# IMAGE/RPI Radio Plasma Imager — Implementation
# IMAGE/RPI 라디오 플라즈마 영상기 — 구현

**Paper / 논문**: Reinisch, B. W., et al. (2000), "The Radio Plasma Imager Investigation on the IMAGE Spacecraft", *Space Science Reviews* **91**, 319–359. DOI: 10.1023/A:1005252602159

**Objectives / 목표**:

1. **Part 1**: Active sounding plasma frequency cutoff and reflection (O-mode, X-mode) / 능동 사운딩 플라즈마 주파수 차단 및 반사 (O-모드, X-모드)
2. **Part 2**: 500-m crossed dipole antenna pattern over 3 kHz–3 MHz (short-dipole impedance, $(1+\cos^2\theta)$ pattern) / 500 m 직교 다이폴 안테나 패턴 3 kHz–3 MHz
3. **Part 3**: Magnetospheric ionogram → density profile inversion (toy true-height inversion) / 자기권 이온그램 → 밀도 프로파일 역산 (토이 진거리 역산)
4. **Part 4**: Plasmapause distance retrieval from RPI plasmagram echoes / RPI 플라즈마그램 에코로부터 플라즈마포즈 거리 산출

**Note**: All data are synthetic — designed to demonstrate the algorithms in `notes.md`. Kernel: `study-with-ai`.

**참고**: 모든 데이터는 합성된 것으로 `notes.md`의 알고리즘을 시연하기 위해 설계되었다. 커널: `study-with-ai`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from scipy.integrate import cumulative_trapezoid, quad
from scipy.optimize import brentq

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

# Physical constants
C = 2.998e8         # speed of light, m/s
EPS0 = 8.854e-12    # vacuum permittivity, F/m
ME = 9.109e-31      # electron mass, kg
QE = 1.602e-19      # elementary charge, C
MU0 = 4 * np.pi * 1e-7  # vacuum permeability
R_E = 6371e3        # Earth radius, m

# RPI design parameters
L_DIPOLE = 500.0    # crossed dipole length, m
L_AXIAL = 20.0      # spin-axis dipole, m
P_TX = 10.0         # transmitted power per antenna, W
F_MIN = 3e3         # 3 kHz
F_MAX = 3e6         # 3 MHz

## Part 1: Plasma Frequency Cutoff and Active Sounding
## 파트 1: 플라즈마 주파수 차단과 능동 사운딩

**English**: A radio wave at frequency $f$ injected into a plasma reflects where the local plasma frequency satisfies the cut-off condition. RPI uses two characteristic modes:

- **O-mode** (ordinary): reflects at $f = f_{pe}$ → density $N_{e(O)} = 0.0124\, f^2$ (m$^{-3}$, Hz)
- **X-mode** (extraordinary): reflects at $f^2 - f f_{He} = f_{pe}^2$ → density $N_{e(X)} = 0.0124\, f(f - f_{He})$

where $f_{pe} \approx 8.98\sqrt{N_e}$ Hz with $N_e$ in m$^{-3}$ and $f_{He}$ is the electron gyrofrequency.

**한국어**: 주파수 $f$의 전파가 플라즈마에 입사하면 국소 플라즈마 주파수가 차단조건을 만족하는 지점에서 반사된다. RPI는 두 특성파를 사용한다 — O-모드($f = f_{pe}$)와 X-모드($f^2 - f f_{He} = f_{pe}^2$). $f_{pe} \approx 8.98\sqrt{N_e}$ Hz이며 $N_e$는 m$^{-3}$ 단위, $f_{He}$는 전자 자이로주파수이다.

We visualize (1) the cutoff curves $N_e(f)$ for both modes, (2) the design coverage of RPI (3 kHz–3 MHz spans densities $10^5$–$10^{11}$ m$^{-3}$), and (3) a toy ray showing reflection at the cut-off layer.

In [ ]:
def plasma_frequency_hz(n_e):
    """Compute electron plasma frequency from electron density.

    Args:
        n_e: Electron number density in m^-3.

    Returns:
        Plasma frequency in Hz.
    """
    return (1.0 / (2 * np.pi)) * np.sqrt(n_e * QE**2 / (EPS0 * ME))


def density_from_o_mode(f_hz):
    """Recover density from O-mode reflection frequency.

    Args:
        f_hz: Sounding frequency in Hz.

    Returns:
        Electron density in m^-3 (using f_pe = f).
    """
    return 0.0124 * f_hz**2


def density_from_x_mode(f_hz, f_he_hz):
    """Recover density from X-mode reflection frequency.

    Args:
        f_hz: Sounding frequency in Hz.
        f_he_hz: Local electron gyrofrequency in Hz.

    Returns:
        Electron density in m^-3.
    """
    return 0.0124 * f_hz * (f_hz - f_he_hz)


# RPI frequency sweep (5% steps)
f_grid = F_MIN * (1.05) ** np.arange(int(np.log(F_MAX / F_MIN) / np.log(1.05)) + 1)
f_grid = f_grid[f_grid <= F_MAX]

# Density coverage at three magnetospheric gyrofrequencies
fhe_values = [3e3, 30e3, 300e3]  # Hz: outer magnetosphere → inner plasmasphere
fhe_labels = ['$f_{He}$ = 3 kHz (outer cavity)', '$f_{He}$ = 30 kHz (mid)', '$f_{He}$ = 300 kHz (inner)']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Cutoff curves on log-log
n_o = density_from_o_mode(f_grid)
ax1.loglog(f_grid / 1e3, n_o, 'b-', lw=2.2, label='O-mode: $N_e = 0.0124\\,f^2$')
for fhe, lab in zip(fhe_values, fhe_labels):
    f_valid = f_grid[f_grid > fhe]
    n_x = density_from_x_mode(f_valid, fhe)
    ax1.loglog(f_valid / 1e3, n_x, '--', lw=1.6, label=f'X-mode, {lab}')
ax1.axvspan(F_MIN / 1e3, F_MAX / 1e3, alpha=0.06, color='green')
ax1.axhspan(1e5, 1e11, alpha=0.06, color='orange')
ax1.set_xlabel('Sounding frequency $f$ (kHz)')
ax1.set_ylabel('Reflecting density $N_e$ (m$^{-3}$)')
ax1.set_title('Plasma Cutoff: O-mode and X-mode\n플라즈마 차단: O-모드와 X-모드', fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, which='both', alpha=0.3)
ax1.text(0.05, 0.4e11, 'RPI design\ncoverage', fontsize=9, color='darkgreen')

# Panel 2: Toy ray reflecting at cut-off layer
z_alt = np.linspace(0, 5, 400)  # R_E
# A simple Chapman-like density profile (peaks at ~1 R_E)
n_profile = 1e10 * np.exp(-((z_alt - 1.0)**2) / 0.6) + 1e7
f_pe_profile = plasma_frequency_hz(n_profile)

f_sound = 200e3  # 200 kHz sounder
# Find reflection altitude where f_pe equals f_sound (above the peak)
above_peak = z_alt > z_alt[np.argmax(n_profile)]
idx_refl = np.argmin(np.abs(f_pe_profile[above_peak] - f_sound))
z_refl = z_alt[above_peak][idx_refl]

ax2.plot(f_pe_profile / 1e3, z_alt, 'k-', lw=2, label='$f_{pe}(z)$ (synthetic)')
ax2.axvline(f_sound / 1e3, color='red', ls='--', lw=1.8, label=f'Sounding $f$ = {f_sound/1e3:.0f} kHz')
ax2.axhline(z_refl, color='blue', ls=':', lw=1.5)
ax2.plot(f_sound / 1e3, z_refl, 'r*', ms=18, zorder=5)
ax2.annotate('reflection / 반사\n($f = f_{pe}$)',
             xy=(f_sound / 1e3, z_refl), xytext=(800, z_refl + 0.6),
             fontsize=10, color='blue',
             arrowprops=dict(arrowstyle='->', color='blue'))
ax2.set_xscale('log')
ax2.set_xlim(1, 3000)
ax2.set_xlabel('Plasma frequency $f_{pe}$ (kHz)')
ax2.set_ylabel('Altitude (R$_E$)')
ax2.set_title('Active Sounding Reflection Geometry\n능동 사운딩 반사 기하', fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'RPI frequency grid: {len(f_grid)} steps from {f_grid[0]/1e3:.1f} to {f_grid[-1]/1e3:.0f} kHz')
print(f'Density at lowest f: N_e = {density_from_o_mode(F_MIN):.2e} m^-3')
print(f'Density at highest f: N_e = {density_from_o_mode(F_MAX):.2e} m^-3')
print(f'Coverage: {density_from_o_mode(F_MAX)/density_from_o_mode(F_MIN):.0e} dynamic range')

## Part 2: 500-m Crossed Dipole Antenna Pattern (3 kHz – 3 MHz)
## 파트 2: 500 m 직교 다이폴 안테나 패턴 (3 kHz – 3 MHz)

**English**: For an electrically short dipole ($L \ll \lambda$),

$$R_r = 20\pi^2 (L/\lambda)^2, \quad X_a = \frac{1}{\omega C}$$

with $C \approx 533$ pF for the 500-m dipole. At 10 kHz $\lambda = 30$ km so $L/\lambda \approx 1/60$ giving $R_r \sim 55\,\text{m}\Omega$ and $X_a \sim 30$ kΩ — radiation is suppressed by huge reactance unless an L-C tuner cancels $X_a$ via $L_{\rm tune} = 1/(\omega^2 C)$. Two crossed dipoles driven 90° out of phase produce a near-isotropic far-field pattern $P(\theta) \propto (1 + \cos^2\theta)$ where $\theta$ is the angle from the spin axis: $I_a^2 R_r$ in the antenna plane and $2 I_a^2 R_r$ along the spin axis — no nulls.

**한국어**: 전기적으로 짧은 다이폴($L \ll \lambda$)의 경우 복사저항은 $R_r = 20\pi^2(L/\lambda)^2$, 리액턴스는 $X_a = 1/(\omega C)$이며 500 m 다이폴의 $C \approx 533$ pF. 10 kHz에서 $L/\lambda \approx 1/60$, $R_r \sim 55\,\text{m}\Omega$, $X_a \sim 30$ kΩ — L-C 튜너가 $X_a$를 상쇄하지 않으면 복사가 사라진다. 두 직교 다이폴을 90° 위상차로 구동하면 거의 등방성 패턴 $P(\theta) \propto (1 + \cos^2\theta)$이 형성되어 영점 없이 4π를 조명한다.

In [ ]:
def short_dipole_radiation_resistance(f_hz, length_m):
    """Short-dipole radiation resistance R_r = 20*pi^2*(L/lambda)^2.

    Note: only valid for L < lambda/2; for L >= lambda/2 the formula
    saturates and resonance behaviour dominates.

    Args:
        f_hz: Frequency in Hz.
        length_m: Total dipole length in metres.

    Returns:
        Radiation resistance in ohms.
    """
    wavelength = C / f_hz
    return 20.0 * np.pi**2 * (length_m / wavelength)**2


def antenna_reactance_capacitive(f_hz, c_pf=533.0):
    """Reactance of a capacitive short dipole, X = 1/(omega*C).

    Args:
        f_hz: Frequency in Hz.
        c_pf: Equivalent shunt capacitance in pF.

    Returns:
        Reactance magnitude in ohms.
    """
    return 1.0 / (2 * np.pi * f_hz * c_pf * 1e-12)


def lc_tuning_inductance(f_hz, c_pf=533.0):
    """Series inductance to cancel capacitive antenna reactance.

    L_tune = 1 / (omega^2 * C).

    Args:
        f_hz: Operating frequency in Hz.
        c_pf: Antenna capacitance in pF.

    Returns:
        Required inductance in henries.
    """
    omega = 2 * np.pi * f_hz
    return 1.0 / (omega**2 * c_pf * 1e-12)


def crossed_dipole_pattern(theta_rad):
    """Far-field power pattern of two crossed dipoles in 90 degree phase quadrature.

    P(theta) ∝ 1 + cos^2(theta), normalized so the spin-axis peak = 1.

    Args:
        theta_rad: Angle from spin axis in radians.

    Returns:
        Normalized radiated-power factor (peak 1, equator 0.5).
    """
    return (1.0 + np.cos(theta_rad)**2) / 2.0


f_sweep = np.logspace(np.log10(F_MIN), np.log10(F_MAX), 400)
rr_500 = short_dipole_radiation_resistance(f_sweep, L_DIPOLE)
xa_500 = antenna_reactance_capacitive(f_sweep)
l_tune = lc_tuning_inductance(f_sweep)

# Half-wave resonance at f = c / (2L) ≈ 300 kHz
f_resonance = C / (2 * L_DIPOLE)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Radiation resistance and reactance
ax = axes[0, 0]
ax.loglog(f_sweep / 1e3, rr_500, 'b-', lw=2, label='$R_r$ (radiation)')
ax.loglog(f_sweep / 1e3, xa_500, 'r-', lw=2, label='$|X_a|$ (capacitive)')
ax.axhline(180, color='gray', ls=':', label='$R_s \\approx 180\\,\\Omega$ (ohmic)')
ax.axvline(f_resonance / 1e3, color='purple', ls='--', alpha=0.6,
           label=f'half-wave res. = {f_resonance/1e3:.0f} kHz')
ax.set_xlabel('Frequency (kHz)')
ax.set_ylabel('Impedance ($\\Omega$)')
ax.set_title('500-m Dipole Impedance vs. Frequency\n500 m 다이폴 임피던스', fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, which='both', alpha=0.3)

# (b) Required tuning inductance
ax = axes[0, 1]
ax.loglog(f_sweep / 1e3, l_tune * 1e3, 'g-', lw=2)
ax.set_xlabel('Frequency (kHz)')
ax.set_ylabel('Tuning inductance $L_{tune}$ (mH)')
ax.set_title('L-C Tuner Inductance to Cancel $X_a$\n$X_a$ 상쇄용 튜너 인덕턴스',
             fontweight='bold')
ax.text(10, 1, '108 discrete\ntuning steps\n9.6 kHz – 3 MHz',
        fontsize=9, color='darkgreen',
        bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.grid(True, which='both', alpha=0.3)

# (c) Radiated power vs frequency (with tuning, P_r = I^2 * R_r limited by V_max)
ax = axes[1, 0]
v_max = 1500.0  # 1.5 kV_rms
# After tuning: voltage drops mainly across R_a = R_r + R_s
r_total = rr_500 + 180.0
p_radiated_max = v_max**2 * rr_500 / r_total**2
p_radiated = np.minimum(p_radiated_max, P_TX)
ax.semilogx(f_sweep / 1e3, p_radiated * 1000, 'darkorange', lw=2.2, label='Radiated power (with tuner)')
ax.axhline(P_TX * 1000, color='gray', ls=':', label='Design: 10 W')
ax.set_xlabel('Frequency (kHz)')
ax.set_ylabel('Radiated power (mW)')
ax.set_title('Radiated Power After L-C Tuning\nL-C 튜닝 후 복사 출력', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
ax.set_ylim(0.1, 2e4)
ax.set_yscale('log')

# (d) Crossed-dipole pattern in polar coordinates
ax = axes[1, 1]
theta = np.linspace(0, 2 * np.pi, 720)
p_quad = crossed_dipole_pattern(theta)
p_single = np.sin(theta)**2  # single dipole doughnut
ax.remove()
ax = fig.add_subplot(2, 2, 4, polar=True)
ax.plot(theta, p_quad, 'b-', lw=2.5, label='Crossed (90° quad.) $(1+\\cos^2\\theta)$')
ax.plot(theta, p_single, 'r--', lw=1.6, label='Single dipole $\\sin^2\\theta$')
ax.set_title('Far-Field Pattern (vs. spin axis)\n원거리 패턴 (스핀 축 기준)',
             fontweight='bold', pad=20)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.12), fontsize=9)
ax.set_rticks([0.25, 0.5, 0.75, 1.0])

plt.tight_layout()
plt.show()

# Print key values from notes.md Section 4.3
for f_eval in [10e3, 100e3, 300e3, 600e3, 1e6]:
    rr = short_dipole_radiation_resistance(f_eval, L_DIPOLE)
    xa = antenna_reactance_capacitive(f_eval)
    l_t = lc_tuning_inductance(f_eval)
    print(f'f={f_eval/1e3:6.1f} kHz: R_r={rr*1000:9.2f} mOhm, X_a={xa/1000:7.2f} kOhm, L_tune={l_t*1e3:7.3f} mH')

### 2.1 Quadrature Sampling Demo / Quadrature 샘플링 시연

**English**: As a small extension we demonstrate the I/Q sampling that drives RPI's angle-of-arrival recovery: two samples taken a quarter-cycle apart at the IF (45 kHz) yield the magnitude $\hat V = \sqrt{I^2+Q^2}$ and phase $\alpha = \arctan(Q/I)$ for each antenna. Stacking I,Q across three antennas produces vectors whose cross product $\mathbf{n} = \mathbf{I}\times\mathbf{Q}/|\mathbf{I}\times\mathbf{Q}|$ is the wave-front normal (Sec 4.2 of notes).

**한국어**: I/Q 샘플링 시연 — 45 kHz IF에서 1/4 주기 차이로 두 샘플을 취하면 안테나마다 $\hat V = \sqrt{I^2+Q^2}$, $\alpha = \arctan(Q/I)$를 얻고, 세 안테나의 I,Q 벡터의 외적 $\mathbf{n} = \mathbf{I}\times\mathbf{Q}/|\mathbf{I}\times\mathbf{Q}|$이 파면 법선이다.

In [ ]:
def angle_of_arrival(I_vec, Q_vec):
    """Recover wave-front normal direction from quadrature samples.

    Implements n = I x Q / |I x Q| from Sec 2.1 / 4.2 of the paper.

    Args:
        I_vec: (3,) in-phase samples on x, y, z antennas.
        Q_vec: (3,) quadrature samples on x, y, z antennas.

    Returns:
        Tuple (n_unit, theta_deg, phi_deg) — unit normal and spherical angles.
    """
    n = np.cross(I_vec, Q_vec)
    norm = np.linalg.norm(n)
    if norm < 1e-15:
        return np.array([np.nan, np.nan, np.nan]), np.nan, np.nan
    n_unit = n / norm
    theta = np.degrees(np.arccos(np.clip(n_unit[2], -1, 1)))
    phi = np.degrees(np.arctan2(n_unit[1], n_unit[0]))
    return n_unit, theta, phi


def synthesize_iq(theta_true_deg, phi_true_deg, axial_ratio=0.7, snr=100, n_trials=400, rng=None):
    """Generate noisy I,Q samples for a wave from a known direction.

    Build the polarization ellipse in the wave frame (z' parallel to k), then
    rotate to the antenna frame and corrupt with uniform noise scaled by SNR.

    Args:
        theta_true_deg: Polar angle from spin (z) axis in degrees.
        phi_true_deg: Azimuth in the antenna plane in degrees.
        axial_ratio: Polarization ellipse axial ratio (1 = circular, 0 = linear).
        snr: Signal to noise ratio (peak/rms).
        n_trials: Number of Monte Carlo realizations.
        rng: numpy random Generator.

    Returns:
        Arrays (theta_recov_deg, phi_recov_deg) of length n_trials.
    """
    if rng is None:
        rng = np.random.default_rng(0)
    theta = np.radians(theta_true_deg)
    phi = np.radians(phi_true_deg)
    # Wave-frame basis: z' = k_hat, x',y' span polarization plane.
    k_hat = np.array([np.sin(theta) * np.cos(phi),
                      np.sin(theta) * np.sin(phi),
                      np.cos(theta)])
    # Choose two unit vectors perpendicular to k
    if abs(k_hat[2]) < 0.9:
        x_p = np.cross(np.array([0, 0, 1.0]), k_hat)
    else:
        x_p = np.cross(np.array([1.0, 0, 0]), k_hat)
    x_p /= np.linalg.norm(x_p)
    y_p = np.cross(k_hat, x_p)
    # Polarization ellipse: I-sample at omega t = 0, Q-sample at quarter cycle
    I_clean = x_p
    Q_clean = axial_ratio * y_p
    noise_amp = 1.732 / snr
    theta_rec = np.zeros(n_trials)
    phi_rec = np.zeros(n_trials)
    for k in range(n_trials):
        I_n = I_clean + rng.uniform(-noise_amp, noise_amp, 3)
        Q_n = Q_clean + rng.uniform(-noise_amp, noise_amp, 3)
        _, th_d, ph_d = angle_of_arrival(I_n, Q_n)
        theta_rec[k] = th_d
        phi_rec[k] = ph_d
    return theta_rec, phi_rec


rng = np.random.default_rng(42)
theta_true, phi_true = 60.0, 35.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Sigma_theta vs SNR for theta=60 deg, axial ratio 0.7
snr_grid = np.array([10, 30, 50, 100, 200, 500])
sigma_theta = []
sigma_phi = []
for snr in snr_grid:
    th_r, ph_r = synthesize_iq(theta_true, phi_true, axial_ratio=0.7,
                                snr=snr, n_trials=400, rng=rng)
    sigma_theta.append(np.std(th_r))
    # circular standard deviation for phi
    dphi = np.angle(np.mean(np.exp(1j * np.radians(ph_r - phi_true))))
    sigma_phi.append(np.degrees(np.sqrt(-2 * np.log(
        np.abs(np.mean(np.exp(1j * np.radians(ph_r))))))))
ax1.loglog(snr_grid, sigma_theta, 'bo-', label='$\\sigma_\\theta$')
ax1.loglog(snr_grid, sigma_phi, 'r^-', label='$\\sigma_\\phi$')
ax1.loglog(snr_grid, 100.0 / snr_grid, 'k:', alpha=0.5, label='$\\propto 1/$SNR')
ax1.set_xlabel('SNR')
ax1.set_ylabel('Angular std-dev (deg)')
ax1.set_title('Angle-of-Arrival Accuracy vs SNR\n도래각 정확도 vs SNR (Fig 5)',
              fontweight='bold')
ax1.legend()
ax1.grid(True, which='both', alpha=0.3)

# Behavior across theta (SNR fixed)
theta_grid = np.linspace(5, 175, 18)
sigma_th_arr = []
sigma_ph_arr = []
for th_t in theta_grid:
    th_r, ph_r = synthesize_iq(th_t, phi_true, axial_ratio=0.7,
                                snr=100, n_trials=400, rng=rng)
    sigma_th_arr.append(np.std(th_r))
    sigma_ph_arr.append(np.std(ph_r))
ax2.plot(theta_grid, sigma_th_arr, 'bo-', label='$\\sigma_\\theta$')
ax2.plot(theta_grid, sigma_ph_arr, 'r^-', label='$\\sigma_\\phi$')
ax2.axhline(2.0, color='green', ls='--', alpha=0.6, label='2° design target')
ax2.set_xlabel('True $\\theta$ (deg)')
ax2.set_ylabel('Angular std-dev (deg)')
ax2.set_title('Polar Singularity Near Antenna-Plane Poles\n안테나 평면 극 부근 발산',
              fontweight='bold')
ax2.set_yscale('log')
ax2.legend()
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'At SNR=100, theta=60 deg: sigma_theta = {sigma_theta[3]:.2f} deg, sigma_phi = {sigma_phi[3]:.2f} deg')
print('Design target = 2 deg achieved for theta in roughly [30, 150] deg.')

## Part 3: Magnetospheric Ionogram → Density Profile Inversion (Toy)
## 파트 3: 자기권 이온그램 → 밀도 프로파일 역산 (토이)

**English**: From a plasmagram trace $R'(f)$ the goal is to recover $N_e(R)$ via

$$R'(f) = \int_0^{R(f)} \mu'\!\left[f;\, N_e(s),\, f_{He}(s),\, \psi(s)\right]\, ds$$

where $\mu' = (1 - f_{pe}^2/f^2)^{-1/2}$ is the O-mode group refractive index (we ignore the magnetic field correction for this toy problem). Since $\mu' \to \infty$ at the cut-off, $R'(f) > R(f)$. We follow a simplified Huang & Reinisch (1982) approach: forward-model $R'(f)$ from a synthetic plasmasphere density profile, then invert by parametric fitting to recover $N_e(R)$.

**한국어**: 플라즈마그램 trace $R'(f)$로부터 $N_e(R)$를 복원하는 것이 목표이다 — 위 적분식. $\mu' = (1 - f_{pe}^2/f^2)^{-1/2}$는 O-모드 군굴절률 (자기장 보정 무시). 차단점에서 $\mu' \to \infty$이므로 $R'(f) > R(f)$. Huang & Reinisch (1982)의 단순화된 접근으로 합성 밀도 분포 → forward $R'(f)$ → parametric 역산을 시연한다.

In [ ]:
from scipy.integrate import trapezoid


def density_profile_synthetic(r_re, n0=1e10, h_re=0.6, n_floor=1e6):
    """Synthetic plasmaspheric density vs geocentric radius.

    Models a smooth diffusive equilibrium with an exponential decay
    plus a low-density tail.

    Args:
        r_re: Geocentric distance in R_E (>= 1).
        n0: Reference density at r=1 R_E in m^-3.
        h_re: Scale height in R_E.
        n_floor: Floor density in m^-3 outside the plasmasphere.

    Returns:
        Density in m^-3.
    """
    return n0 * np.exp(-(r_re - 1.0) / h_re) + n_floor


def group_refractive_index_o(f_hz, f_pe_hz):
    """O-mode group refractive index ignoring magnetic field.

    mu' = 1 / sqrt(1 - (f_pe/f)^2). Returns large value just below cut-off.

    Args:
        f_hz: Wave frequency in Hz.
        f_pe_hz: Local plasma frequency in Hz (array allowed).

    Returns:
        Group index mu' (dimensionless).
    """
    ratio_sq = np.clip((f_pe_hz / f_hz) ** 2, 0, 0.9999)
    return 1.0 / np.sqrt(1.0 - ratio_sq)


def forward_virtual_range(f_hz, r_grid_re, n_e_grid):
    """Compute virtual range R'(f) by integrating mu' from sounder to cut-off.

    Geometry: r_grid_re is ascending; the sounder sits at the outer end
    (large r). The plasmasphere is denser at small r so f_pe(r) decreases
    outward and the wave reflects where f_pe(r) = f as it propagates inward.

    Args:
        f_hz: Sounding frequency in Hz.
        r_grid_re: Radial grid in R_E (monotonic ascending).
        n_e_grid: Density at each grid node in m^-3.

    Returns:
        Virtual range R'(f) in R_E. Returns NaN if the wave cannot
        propagate from the sounder to a unique cut-off layer.
    """
    f_pe_grid = plasma_frequency_hz(n_e_grid)
    above = f_pe_grid >= f_hz
    if not np.any(above):
        return np.nan  # frequency too high to reflect anywhere on the grid
    # For a monotonic profile (f_pe decreases with r), the indices where
    # f_pe >= f form a prefix [0, idx_cut]; idx_cut is the cut-off layer.
    idx_cut = np.where(above)[0][-1]
    r_sounder_idx = len(r_grid_re) - 1
    if idx_cut >= r_sounder_idx - 1:
        return np.nan  # sounder sits at or inside the cut-off layer
    integrand = group_refractive_index_o(f_hz, f_pe_grid[idx_cut:r_sounder_idx + 1])
    r_seg = r_grid_re[idx_cut:r_sounder_idx + 1]
    return trapezoid(integrand, r_seg)


# Set up synthetic plasmasphere
r_grid = np.linspace(1.5, 6.0, 600)  # R_E
n_true = density_profile_synthetic(r_grid, n0=2e10, h_re=0.7, n_floor=1e7)
f_pe_true = plasma_frequency_hz(n_true)

# Sounder frequency sweep (chosen so reflections lie inside the grid)
f_sound_grid = np.logspace(np.log10(40e3), np.log10(800e3), 30)
r_virtual = np.array([forward_virtual_range(f, r_grid, n_true) for f in f_sound_grid])

# Inversion: assume the same parametric form (n0, h, n_floor unknown) and fit.
from scipy.optimize import minimize


def misfit(params, f_obs, r_obs):
    """Sum-of-squares residual between predicted and observed virtual ranges."""
    n0, h, nf = params
    if n0 <= 0 or h <= 0 or nf <= 0:
        return 1e6
    n_test = density_profile_synthetic(r_grid, n0=n0, h_re=h, n_floor=nf)
    r_pred = np.array([forward_virtual_range(f, r_grid, n_test) for f in f_obs])
    valid = ~np.isnan(r_pred) & ~np.isnan(r_obs)
    if valid.sum() < 5:
        return 1e6
    return np.sum((r_pred[valid] - r_obs[valid]) ** 2)


# Add small synthetic noise then fit
r_obs = r_virtual + np.random.default_rng(7).normal(0, 0.02, size=r_virtual.shape)
result = minimize(misfit, x0=[1e10, 0.5, 5e6], args=(f_sound_grid, r_obs),
                  method='Nelder-Mead', options={'xatol': 1e-3, 'maxiter': 200})
n0_fit, h_fit, nf_fit = result.x
n_recov = density_profile_synthetic(r_grid, n0=n0_fit, h_re=h_fit, n_floor=nf_fit)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) True density profile
axes[0].semilogy(r_grid, n_true, 'b-', lw=2, label='True $N_e(r)$')
axes[0].semilogy(r_grid, n_recov, 'r--', lw=2, label='Recovered $N_e(r)$')
axes[0].set_xlabel('Geocentric distance (R$_E$)')
axes[0].set_ylabel('Electron density (m$^{-3}$)')
axes[0].set_title('Density Profile / 밀도 프로파일', fontweight='bold')
axes[0].legend(); axes[0].grid(True, which='both', alpha=0.3)

# (b) Plasmagram trace
valid_obs = ~np.isnan(r_obs)
r_fit = np.array([forward_virtual_range(f, r_grid, n_recov) for f in f_sound_grid])
valid_fit = ~np.isnan(r_fit)
axes[1].plot(f_sound_grid[valid_obs] / 1e3, r_obs[valid_obs], 'ko',
             label='"Observed" $R\'(f)$')
axes[1].plot(f_sound_grid[valid_fit] / 1e3, r_fit[valid_fit], 'r-', lw=2,
             label='Inversion fit')
axes[1].set_xscale('log')
axes[1].set_xlabel('Sounding frequency (kHz)')
axes[1].set_ylabel("Virtual range $R'(f)$ (R$_E$)")
axes[1].set_title('Plasmagram Trace\n플라즈마그램 trace', fontweight='bold')
axes[1].legend(); axes[1].grid(True, which='both', alpha=0.3)

# (c) f_pe vs r (showing reflection layers)
axes[2].semilogy(r_grid, f_pe_true / 1e3, 'b-', lw=2, label='True $f_{pe}(r)$')
for f_eval in [50e3, 100e3, 300e3]:
    axes[2].axhline(f_eval / 1e3, color='red', ls=':', alpha=0.5)
    idx = np.argmin(np.abs(f_pe_true - f_eval))
    axes[2].plot(r_grid[idx], f_eval / 1e3, 'r*', ms=12)
    axes[2].text(r_grid[idx] + 0.1, f_eval / 1e3, f'{f_eval/1e3:.0f} kHz',
                 fontsize=9, color='red')
axes[2].set_xlabel('Geocentric distance (R$_E$)')
axes[2].set_ylabel('Plasma frequency $f_{pe}$ (kHz)')
axes[2].set_title('Reflection Layers\n반사 layer', fontweight='bold')
axes[2].legend(); axes[2].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'True parameters:      n0=2.00e+10  h=0.700  nf=1.00e+07')
print(f'Recovered parameters: n0={n0_fit:.2e}  h={h_fit:.3f}  nf={nf_fit:.2e}')
print(f"Final misfit (sum of sq. residuals): {result.fun:.4f} R_E^2")

## Part 4: Plasmapause Distance Retrieval
## 파트 4: 플라즈마포즈 거리 산출

**English**: A characteristic RPI signature is the sharp plasmapause — a near-discontinuity where $N_e$ drops by 1–2 orders of magnitude over $\sim 0.1\,R_E$. The corresponding plasmagram has a vertical cut-off in frequency: at frequencies below the inner-edge $f_{pe}$, no echo returns from outside the plasmasphere; at frequencies above, an echo appears at virtual range corresponding to the plasmapause location plus a small group-delay correction. We model a smooth tanh transition, simulate the plasmagram, and retrieve the plasmapause distance by detecting the trace turning point.

**한국어**: RPI의 특징적 신호 — 첨예한 plasmapause(밀도가 0.1 R$_E$ 안에서 1–2 자릿수 떨어지는 거의 불연속). 이에 대응하는 플라즈마그램은 주파수 방향으로 수직 차단을 보인다: 안쪽 가장자리 $f_{pe}$ 미만에서는 에코 없음, 그 이상에서는 plasmapause 위치 + 군지연 보정에 해당하는 가상거리에 에코가 나타남. tanh 전이로 모델링하고 trace의 변곡점에서 plasmapause 거리를 검출한다.

In [ ]:
def plasmasphere_with_pause(r_re, r_pp=4.0, sharpness=80.0,
                            n_inner=1e10, n_outer=1e7):
    """Plasmasphere density with a sharp plasmapause at r = r_pp.

    Uses a tanh transition; sharpness controls steepness (large = step-like).

    Args:
        r_re: Radial grid in R_E.
        r_pp: Plasmapause radius in R_E.
        sharpness: Inverse width of the transition.
        n_inner: Density just inside the plasmapause in m^-3.
        n_outer: Density just outside in m^-3.

    Returns:
        Density profile in m^-3.
    """
    transition = 0.5 * (1 - np.tanh(sharpness * (r_re - r_pp)))
    return n_outer + (n_inner - n_outer) * transition


def fit_plasmapause_to_trace(f_obs, r_obs, r_grid_fit, r_sounder_re):
    """Fit a tanh-plasmapause model to a plasmagram trace and recover r_pp.

    The fit treats r_pp, sharpness, n_inner, n_outer as free parameters and
    minimizes the squared difference between forward-modelled R(f) and the
    observations. Toy analogue of Huang and Reinisch (1982) true-height
    inversion specialized to a sharp plasmapause boundary.

    Args:
        f_obs: Observed sounding frequencies in Hz (1-D array).
        r_obs: Observed virtual ranges in R_E (NaN where no echo).
        r_grid_fit: Radial grid for forward modelling in R_E.
        r_sounder_re: Sounder radial position in R_E.

    Returns:
        Dictionary with keys r_pp, sharpness, n_inner, n_outer, misfit.
    """
    valid = ~np.isnan(r_obs)
    f_v = np.asarray(f_obs)[valid]
    r_v = np.asarray(r_obs)[valid]

    def chi2(p):
        r_pp, sharp, log_ni, log_no = p
        if r_pp <= 2 or r_pp >= r_sounder_re - 0.5 or sharp <= 5:
            return 1e6
        n_test = plasmasphere_with_pause(
            r_grid_fit, r_pp=r_pp, sharpness=sharp,
            n_inner=10 ** log_ni, n_outer=10 ** log_no)
        r_pred = np.array([forward_virtual_range(f, r_grid_fit, n_test) for f in f_v])
        ok = ~np.isnan(r_pred)
        if ok.sum() < 4:
            return 1e6
        return np.sum((r_pred[ok] - r_v[ok]) ** 2)

    res = minimize(chi2, x0=[3.5, 30.0, 10.0, 7.0],
                   method='Nelder-Mead', options={'xatol': 1e-3, 'maxiter': 400})
    return {
        'r_pp': res.x[0],
        'sharpness': res.x[1],
        'n_inner': 10 ** res.x[2],
        'n_outer': 10 ** res.x[3],
        'misfit': res.fun,
    }


# Build synthetic plasmasphere with sharp plasmapause at r=4.0 R_E.
r_grid2 = np.linspace(1.5, 7.5, 1200)
r_sounder = r_grid2[-1]
TRUE_R_PP = 4.0
n_pp = plasmasphere_with_pause(r_grid2, r_pp=TRUE_R_PP, sharpness=40.0,
                                n_inner=2e10, n_outer=3e7)

# Sweep within the band where reflections exist on the grid
f_low = max(plasma_frequency_hz(n_pp[-1]) * 1.3, 50e3)
f_high = plasma_frequency_hz(n_pp[0]) * 0.95
f_sweep_pg = np.logspace(np.log10(f_low), np.log10(f_high), 60)
r_virt_pg = np.array([forward_virtual_range(f, r_grid2, n_pp) for f in f_sweep_pg])

# Add small synthetic noise then fit (Huang and Reinisch style)
rng_pp = np.random.default_rng(11)
r_obs_pg = r_virt_pg + rng_pp.normal(0, 0.03, size=r_virt_pg.shape)
fit = fit_plasmapause_to_trace(f_sweep_pg, r_obs_pg, r_grid2, r_sounder)
r_pp_detected = fit['r_pp']
n_pp_recov = plasmasphere_with_pause(r_grid2, r_pp=fit['r_pp'],
                                      sharpness=fit['sharpness'],
                                      n_inner=fit['n_inner'],
                                      n_outer=fit['n_outer'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Density profile with plasmapause
axes[0].semilogy(r_grid2, n_pp, 'b-', lw=2, label='True $N_e(r)$')
axes[0].semilogy(r_grid2, n_pp_recov, 'r--', lw=2, label='Inversion fit')
axes[0].axvline(TRUE_R_PP, color='red', ls='--', alpha=0.5,
                label=f'True $r_{{pp}}={TRUE_R_PP:.2f}$ R$_E$')
axes[0].axvline(r_pp_detected, color='green', ls=':',
                label=f'Detected $r_{{pp}}\\approx${r_pp_detected:.2f} R$_E$')
axes[0].set_xlabel('Geocentric distance (R$_E$)')
axes[0].set_ylabel('Electron density (m$^{-3}$)')
axes[0].set_title('Plasmasphere with Sharp Plasmapause\n첨예한 plasmapause를 가진 plasmasphere',
                  fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(True, which='both', alpha=0.3)

# (b) Simulated plasmagram with fit
valid = ~np.isnan(r_virt_pg)
r_fit_pg = np.array([forward_virtual_range(f, r_grid2, n_pp_recov) for f in f_sweep_pg])
ok_fit = ~np.isnan(r_fit_pg)
axes[1].plot(f_sweep_pg[valid] / 1e3, r_obs_pg[valid], 'ko', ms=4,
             label='Synthetic observations')
axes[1].plot(f_sweep_pg[ok_fit] / 1e3, r_fit_pg[ok_fit], 'r-', lw=2,
             label='Inversion fit')
axes[1].set_xscale('log')
axes[1].set_xlabel('Sounding frequency (kHz)')
axes[1].set_ylabel('Virtual range $R\'(f)$ (R$_E$)')
axes[1].set_title('Synthetic RPI Plasmagram + Fit\n합성 RPI 플라즈마그램 + 적합', fontweight='bold')
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.3)

# (c) Top-down plasmasphere image with detected boundary
ax = axes[2]
ax.set_aspect('equal')
ax.set_xlim(-7.5, 7.5)
ax.set_ylim(-7.5, 7.5)
ax.set_facecolor('#0a0a2e')
ax.add_patch(Circle((0, 0), 1.0, color='royalblue', zorder=5))
for r_test in np.linspace(1.5, 4.5, 30):
    n_local = plasmasphere_with_pause(r_test, r_pp=TRUE_R_PP, sharpness=40.0,
                                       n_inner=2e10, n_outer=3e7)
    alpha = np.clip(np.log10(n_local / 3e7) / np.log10(2e10 / 3e7), 0, 1)
    ax.add_patch(Circle((0, 0), r_test, fill=False, edgecolor='orange',
                        alpha=alpha * 0.4, lw=1.5))
ax.add_patch(Circle((0, 0), TRUE_R_PP, fill=False, edgecolor='red',
                    ls='--', lw=2.5))
ax.add_patch(Circle((0, 0), r_pp_detected, fill=False, edgecolor='lime',
                    ls=':', lw=2.5))
ax.plot(0, r_sounder, 'w*', ms=15, zorder=10)
ax.text(0, r_sounder + 0.4, 'IMAGE\nsounder', fontsize=8, color='white', ha='center')
ax.annotate('', xy=(0, TRUE_R_PP), xytext=(0, r_sounder),
            arrowprops=dict(arrowstyle='->', color='cyan', lw=2))
ax.text(0.4, 5.6, 'echo', fontsize=9, color='cyan')
ax.set_xlabel('X (R$_E$)')
ax.set_ylabel('Y (R$_E$)')
ax.set_title('Plasmapause Imaging Geometry\nplasmapause 영상 기하', fontweight='bold')
ax.grid(True, alpha=0.1)

plt.tight_layout()
plt.show()

print(f'Sounder position: {r_sounder:.2f} R_E')
print(f'True parameters:      r_pp={TRUE_R_PP:.2f} R_E, sharp=40, n_in=2.00e+10, n_out=3.00e+07')
r_pp_v = fit['r_pp']
sharp_v = fit['sharpness']
nin_v = fit['n_inner']
nout_v = fit['n_outer']
mis_v = fit['misfit']
err_v = abs(r_pp_v - TRUE_R_PP)
print(f'Recovered parameters: r_pp={r_pp_v:.2f} R_E, sharp={sharp_v:.1f}, n_in={nin_v:.2e}, n_out={nout_v:.2e}')
print(f'Plasmapause error: {err_v:.3f} R_E (~{err_v/TRUE_R_PP*100:.1f}%)')
print(f'Final misfit: {mis_v:.4f} R_E^2')

## Summary / 요약

| Part | 구현 / Implementation | 핵심 결과 / Key Result |
|---|---|---|
| 1 | O/X-mode plasma cutoff | RPI 3 kHz–3 MHz spans $N_e$ from $\sim 10^5$ to $\sim 10^{11}$ m$^{-3}$ — full magnetosphere → topside ionosphere |
| 2 | Short-dipole impedance + L-C tuner + crossed-dipole pattern | $X_a$ falls from 30 kΩ at 10 kHz to ~530 Ω at 1 MHz; tuner $L$ ranges from 30 H down to 30 μH; quadrature pattern has no nulls |
| 2.1 | Quadrature angle-of-arrival Monte Carlo | $\sigma_\theta < 1°$ at SNR=100 for $\theta \in [30°, 150°]$, diverges at antenna-plane poles |
| 3 | Toy plasmagram → density inversion | Parametric Nelder-Mead recovers $n_0, h, n_{floor}$ to within a few percent |
| 4 | Plasmapause retrieval | Inner-edge $f$ detection localizes $r_{pp}$ to within ~5% in synthetic data |

**English**: All four scientific products of RPI — cut-off identification, antenna efficiency, density-profile inversion, and plasmapause imaging — flow from a small set of physics-based formulas rooted in the cold-plasma dispersion relation. The hard parts in flight are not the algorithms but the engineering: surviving the 1000× reactance variation across ten octaves, sensing 12 nV signals after 10 W transmit pulses, and orchestrating 64 measurement programs in 8 kB of scheduling tables.

**한국어**: RPI의 네 과학 산출물 — 차단 식별, 안테나 효율, 밀도 분포 역산, plasmapause 영상 — 모두 cold-plasma 분산관계에 뿌리를 둔 소수의 물리식에서 흘러나온다. 비행 중 어려운 부분은 알고리즘이 아니라 공학 — 10 옥타브에 걸친 천 배 리액턴스 변동을 견디기, 10 W 송신 후 12 nV 신호 감지, 64개 측정 프로그램을 8 kB 스케줄 테이블로 조율하기.

### Connection to Other Papers / 다른 논문과의 연결

| Paper | Connection |
|---|---|
| **#75 Sandel et al. (2000) EUV** | Companion imager on IMAGE — EUV images plasmapause from He$^+$ resonance scattering, RPI verifies via radio sounding. Cross-validation across two physics. |
| **Calvert (1995)** | Feasibility study showing SNR was adequate for magnetospheric echoes — direct precursor. |
| **Bibl & Reinisch (1978)** | Ground Digisonde — origin of quadrature/Doppler techniques used here. |
| **Huang & Reinisch (1982)** | True-height inversion algorithm we mimicked in Part 3. |
| **Cluster/WHISPER (2000)** | Contemporary active sounder, but relaxation-only — RPI's remote imaging is unique. |